<a href="https://colab.research.google.com/github/DongAnYu/ai-tutor-rag-system/blob/main/notebooks/12-Improve_Query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [2]:
!pip install -q llama-index==0.14.0 openai==1.107.0 llama-index-finetuning==0.4.1 llama-index-embeddings-huggingface==0.6.1 \
                llama-index-embeddings-cohere==0.6.1 cohere==5.18.0 llama-index-readers-web==0.5.3 chromadb==1.0.21 jedi==0.19.2 \
                llama-index-vector-stores-chroma==0.5.3 llama-index-llms-google-genai==0.5.0 google-genai==1.38.0 llama-index-llms-openai==0.5.6

In [3]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR-OPENAI-API-KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR-GOOGLE-API-KEY>"

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')

In [4]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

nest_asyncio.apply()

# Load a Model


In [5]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort':'minimal'})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Load Indexes


In [6]:
from huggingface_hub import hf_hub_download

vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vectorstore.zip:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

In [7]:
!unzip -o vectorstore.zip

Archive:  vectorstore.zip
   creating: ai_tutor_knowledge/
   creating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/length.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/index_metadata.pickle  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/link_lists.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/header.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/data_level0.bin  
  inflating: ai_tutor_knowledge/chroma.sqlite3  


In [8]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Create the vector index
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
vector_index = VectorStoreIndex.from_vector_store(vector_store)

# Multi-Step Query Engine


## GPT-5-mini


In [10]:
from llama_index.core.indices.query.query_transform.base import (
    StepDecomposeQueryTransform,
)

step_decompose_transform_gpt5 = StepDecomposeQueryTransform(verbose=True, llm=Settings.llm) # Uses the LLM to break it into multiple simpler sub-questions (steps)

In [11]:
from llama_index.core.query_engine.multistep_query_engine import MultiStepQueryEngine

#Default query engine
query_engine_gpt5_mini = vector_index.as_query_engine()

# Multi Step Query Engine
multi_step_query_engine = MultiStepQueryEngine(
    query_engine = query_engine_gpt5_mini,
    query_transform = step_decompose_transform_gpt5,
    index_summary = "Used to answer the Questions about RAG, Machine Learning, Deep Learning, and Generative AI, Note: Don't repeat the Same quesion",
)

# Query Dataset

## Default

In [12]:
# Default query engine
query_engine = vector_index.as_query_engine()
res = query_engine.query("Write about Llama 3.1 Model, BERT and PEFT methods")
print(res.response)

I can only answer using the provided excerpts. The excerpts discuss LLaMA (not Llama 3.1 specifically), BERT is not mentioned in the excerpts, and PEFT methods are covered primarily for LLaMA. Based on those excerpts, here is what can be said:

- LLaMA model family (classes and tooling)
  - Configuration and model classes available include LlamaConfig, LlamaTokenizer / LlamaTokenizerFast (with utilities such as build_inputs_with_special_tokens, get_special_tokens_mask, create_token_type_ids_from_sequences, save_vocabulary, and update_post_processor for the fast tokenizer), and model variants like LlamaModel, LlamaForCausalLM, LlamaForSequenceClassification, LlamaForQuestionAnswering, and LlamaForTokenClassification (each exposing a forward method). Flax implementations are provided as FlaxLlamaModel and FlaxLlamaForCausalLM (with __call__).

- PEFT methods for LLaMA in the excerpts
  - LoRA (Low-Rank Adaptation) is highlighted as a PEFT technique used to fine-tune LLaMA; a notebook dem

In [13]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 781b7b12-eca2-47c0-a66e-9d6be670e951
Title	 LLaMA
Text	 on how to fine-tune LLaMA model using LoRA method via the 🤗 PEFT library with intuitive UI. 🌎 - A [notebook](https://github.com/aws/amazon-sagemaker-examples/blob/main/introduction_to_amazon_algorithms/jumpstart-foundation-models/text-generation-open-llama.ipynb) on how to deploy Open-LLaMA model for text generation on Amazon SageMaker. 🌎 ## LlamaConfig[[autodoc]] LlamaConfig## LlamaTokenizer[[autodoc]] LlamaTokenizer    - build_inputs_with_special_tokens    - get_special_tokens_mask    - create_token_type_ids_from_sequences    - save_vocabulary## LlamaTokenizerFast[[autodoc]] LlamaTokenizerFast    - build_inputs_with_special_tokens    - get_special_tokens_mask    - create_token_type_ids_from_sequences    - update_post_processor    - save_vocabulary## LlamaModel[[autodoc]] LlamaModel    - forward## LlamaForCausalLM[[autodoc]] LlamaForCausalLM    - forward## LlamaForSequenceClassification[[autodoc]] LlamaForSequenceClassif

Normal query engine behavior (single-step RAG) Summarization:
The model says: “I can only answer using the provided excerpts…”

This is classic RAG behavior:


*   It only uses retrieved chunks
*   It refuses to hallucinate about: “Llama 3.1” (not in your index) and “BERT” (not in your index)

So the answer is faithful to retrieved context but limited by recall


## GPT-5-mini Multi-Step


In [14]:
response = multi_step_query_engine.query("Write about Llama 3.1 Model, BERT and PEFT methods")
print(response.response)

> Current query: Write about Llama 3.1 Model, BERT and PEFT methods
> New query: Which key features, strengths, and typical uses distinguish Llama 3.1, BERT, and PEFT methods from each other?
> Current query: Write about Llama 3.1 Model, BERT and PEFT methods
> New query: Which key features, strengths, and typical uses distinguish Llama 3.1, BERT, and PEFT methods from each other?
> Current query: Write about Llama 3.1 Model, BERT and PEFT methods
> New query: Which key features, strengths, and typical uses distinguish Llama 3.1, BERT, and PEFT methods from each other?
Llama 3.1 model, BERT, and PEFT methods occupy different parts of the modern ML stack: large generative foundation models, encoder-style pretrained models for understanding, and parameter-efficient adaptation techniques. Below are their key features, strengths, and typical uses.

Llama 3.1
- Key features:
  - Large autoregressive foundation model family optimized for generation and coding tasks; includes specialized vari

In [15]:
for query, response in response.metadata['sub_qa']:
    print(f"**{query}**\n{response}\n")

**Which key features, strengths, and typical uses distinguish Llama 3.1, BERT, and PEFT methods from each other?**
Llama 3.1, BERT, and PEFT occupy different roles in the model/tooling ecosystem and are distinguished by their architectures, primary strengths, and typical use cases:

- Llama 3.1
  - Key features: Large autoregressive foundation model family optimized for generation and coding tasks (variants include specialized models and sizes). Supports long contexts and advanced capabilities such as instruction following and code infilling in specialized variants.
  - Strengths: Strong generative and code-related performance, competitive on coding benchmarks; scalable across multiple parameter sizes; suitable for tasks requiring fluent text or code generation, long-context reasoning, and zero-shot instruction following.
  - Typical uses: Code generation and completion, instruction-following generation, tasks that need large-context generation and high-quality natural-language or code

In [16]:
for src in response.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 9343ec8a-73cd-4bc4-bf80-d4b031edae51
Text	 ><div class="w-full text-center bg-gradient-to-br from-indigo-400 to-indigo-500 rounded-lg py-1.5 font-semibold mb-5 text-white text-lg leading-relaxed">How-to guides</div>      <p class="text-gray-700">Practical guides demonstrating how to apply various PEFT methods across different types of tasks like image classification, causal language modeling, automatic speech recognition, and more. Learn how to use 🤗 PEFT with the DeepSpeed and Fully Sharded Data Parallel scripts.</p>    </a>    <a class="!no-underline border dark:border-gray-700 p-5 rounded-lg shadow hover:shadow-lg" href="./conceptual_guides/lora"      ><div class="w-full text-center bg-gradient-to-br from-pink-400 to-pink-500 rounded-lg py-1.5 font-semibold mb-5 text-white text-lg leading-relaxed">Conceptual guides</div>      <p class="text-gray-700">Get a better theoretical understanding of how LoRA and various soft prompting methods help reduce the number of trainable par

# MultiStepQueryEngine – Query Decomposition Behavior Summary

## Observed Behavior

During the multi-step execution, the following pattern occurred (repeated three times):

Current query: Write about Llama 3.1 Model, BERT and PEFT methods
New query: Which key features, strengths, and typical uses distinguish Llama 3.1, BERT, and PEFT methods from each other?

yaml
Copy code

This reveals how `StepDecomposeQueryTransform` handled the query internally.

---

## StepDecomposeQueryTransform Behavior

Instead of decomposing the original query into multiple atomic sub-questions (e.g., one for Llama 3.1, one for BERT, one for PEFT), the transformer:

- Recognized the query as a **comparison-style question**
- Rewrote it into a more analytical form:

> “Which key features, strengths, and typical uses distinguish Llama 3.1, BERT, and PEFT methods from each other?”

However:

- ❌ It did **not** generate multiple sub-questions  
- ❌ It repeatedly rewrote the same query  
- ❌ No true multi-hop reasoning occurred  

---

## Failure Mode: Multi-Rewrite Instead of Multi-Hop

In this case, multi-step execution degenerated into **multi-rewrite**, not true multi-hop decomposition.

Possible causes:

- The transformer failed to generate multiple steps  
- The prompt or `index_summary` biased the model toward rewriting  
- LlamaIndex fallback logic retried decomposition with the same rewritten query  

---

## Key Insight

`MultiStepQueryEngine` relies heavily on the LLM’s ability to perform **explicit planning**.

If the model only rewrites instead of decomposing, the system:

- Performs repeated single-step retrieval  
- Does not isolate individual entities (Llama / BERT / PEFT)  
- Loses the benefits of true multi-hop RAG  

---

## Conclusion

This run demonstrates a known limitation of `StepDecomposeQueryTransform`:

> **MultiStepQueryEngine can collapse into query rewriting rather than true multi-question decomposition, depending on model behavior and prompt configuration.**

For reliable multi-entity or multi-hop reasoning, prefer:

- `SubQuestionQueryEngine` (explicit sub-question generation), or  
- A dedicated planner model with stronger decomposition control.

# Test gemini-2.5-flash Multi-Step


In [21]:
from llama_index.core.indices.query.query_transform.base import StepDecomposeQueryTransform
from llama_index.core.query_engine.multistep_query_engine import MultiStepQueryEngine
from llama_index.llms.google_genai import GoogleGenAI

llm_gemini = GoogleGenAI(model="gemini-2.5-flash")

step_decompose_transform = StepDecomposeQueryTransform(llm=llm_gemini, verbose=True)

query_engine_gemini = vector_index.as_query_engine(llm=llm_gemini)

query_engine_gemini = MultiStepQueryEngine(
    query_engine=query_engine_gemini,
    query_transform=step_decompose_transform,
    index_summary="Used to answer the Questions about RAG, Machine Learning, Deep Learning,LLMs and Generative AI",
)

In [25]:
response_gemini = query_engine_gemini.query("Write about Llama 3.1 Model, BERT and PEFT")

> Current query: Write about Llama 3.1 Model, BERT and PEFT
> New query: What is Llama 3.1 Model?


> Current query: Write about Llama 3.1 Model, BERT and PEFT
> New query: What are the key features and capabilities of Llama 3.1?
> Current query: Write about Llama 3.1 Model, BERT and PEFT
> New query: None


In [26]:
response_gemini.response

'Llama 3.1\n- Overview: Llama 3.1 is a large open-source AI model from Meta, trained on over 15 trillion tokens using more than 16,000 H100 GPUs. It is the largest and most advanced in its family, available in variants such as 405B, 70B, and 8B (the 8B is recommended for local use via Ollama).\n- Strengths: It provides strong language modeling, mathematical and complex reasoning, and long-text processing. It achieves high scores in zero scrolls quality and in benchmark comparisons, outperforming several contemporaries in areas like mathematical reasoning, complex reasoning, multilingual support, and long-context tasks.\n- Capabilities: Llama 3.1 supports a 128K context length, improved reasoning and coding abilities (generating high-quality code with solid syntax and logic understanding), and zero-shot tool use with potential for agentic behaviors. Its pretraining includes roughly 50% multilingual tokens, enabling robust multilingual processing.\n- Multimodality and limitations: Multim

Gemini generated a very detailed description of Llama 3.1, including:

*   Training scale (15T tokens, 16k+ H100 GPUs)

*   Context length (128K)

*   Benchmarks and comparisons

*   Multilingual coverage and reasoning abilities

However:

*   The vector index did not contain Llama 3.1 or BERT documents

*   Only PEFT / Code-Llama related sources existed

As a result:

*   Llama 3.1 content came from Gemini’s pretrained knowledge, not retrieval

*   BERT and PEFT sections were marked as “no direct information available”

This shows:

*   Gemini prioritizes completeness and reasoning

*   But does not strictly enforce RAG grounding

*   Hallucination risk increases when index coverage is missing

## Test Retriever on Multistep


In [22]:
# import llama_index
# from llama_index.core.indices.query.schema import QueryBundle

# t = QueryBundle("How Retrieval Augmented Generation (RAG) work?")
# query_engine_gemini.retrieve(t)

NotImplementedError: This query engine does not support retrieve, use query directly

## Subquestion Query Engine

agent-style multi-hop RAG architecture:

User query

   ↓

LLMQuestionGenerator (planner)

   ↓
   
[Q1, Q2, Q3, ... Qn]

   ↓

Each Qi → QueryEngineTool (retriever + RAG)

   ↓

Answers A1, A2, A3, ...

   ↓

Final synthesis


In [27]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.question_gen.llm_generators import LLMQuestionGenerator
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort':'minimal'})

question_gen = LLMQuestionGenerator.from_defaults(llm=llm)  # generates multiple atomic sub-questions

query_engine = vector_index.as_query_engine()  # wraps your vector index as a callable tool

query_engine_tools = [
    QueryEngineTool(
        query_engine=query_engine,
        metadata=ToolMetadata(
            name="LlamaIndex",
            description="Used to answer the Questions about RAG, Machine Learning, Deep Learning, and Generative AI. Note: Don't repeat the Same question",
        ),
    ),
]

sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=query_engine_tools,
    question_gen=question_gen,
    use_async=True,
)

response = sub_question_engine.query("Write about Llama 3.1 Model, BERT and PEFT")


Generated 7 sub questions.
[LlamaIndex] Q: What are the key features and capabilities of Llama 3.1?
[LlamaIndex] Q: What are the main architectural details and typical use cases of BERT?
[LlamaIndex] Q: What is PEFT (parameter-efficient fine-tuning), how does it work, and what are common techniques?
[LlamaIndex] Q: How does Llama 3.1 compare to BERT in terms of architecture, training objectives, and typical applications?
[LlamaIndex] Q: How can PEFT methods be applied to Llama 3.1 and to BERT, and what are the benefits and trade-offs for each?
[LlamaIndex] Q: What are practical examples or case studies demonstrating PEFT on models like Llama 3.1 and BERT?
[LlamaIndex] Q: What are best practices and recommended tools/libraries for implementing PEFT with Llama 3.1 and BERT?
[LlamaIndex] A: The provided information discusses Llama 3.1 (a large, recent open-source Transformer-based generative model) but does not provide details about BERT. Because the excerpts do not describe BERT’s archit

In [28]:
response.response

'Llama 3.1, BERT, and PEFT (parameter-efficient fine-tuning) are three complementary pieces of modern NLP tooling: Llama 3.1 is a large, generative transformer model family; BERT is a bidirectional encoder transformer geared toward discriminative/representation tasks; and PEFT is a set of methods and tooling to adapt large pretrained models to tasks while updating only a small number of parameters. Below are concise summaries of each and how they relate.\n\nLlama 3.1 — key characteristics and capabilities\n- Model family and sizes: Offered as a large-scale family with multiple sizes, including a 405 billion–parameter flagship and smaller 70B and 8B variants. Smaller variants are more accessible for local or smaller-cloud deployments.\n- Training scale: The largest variant was trained on trillions of tokens using massive GPU fleets.\n- Long context: Supports very long context windows (examples include up to ~128K tokens), enabling strong long-text understanding and generation.\n- Strong

# SubQuestionQueryEngine – Tool-based Multi-Hop RAG (LlamaIndex)

## Architecture

- Uses `LLMQuestionGenerator` to explicitly decompose a complex query into multiple atomic sub-questions.  
- Each sub-question is executed via a `QueryEngineTool` wrapping the vector index.  
- `SubQuestionQueryEngine` orchestrates:
  - Sub-question generation  
  - Parallel retrieval and answering  
  - Final answer synthesis  

This implements an explicit **planner → tool execution → synthesis** multi-hop RAG pipeline.

---

## Strengths

- True multi-hop decomposition with explicit planning  
- Parallel (async) execution reduces end-to-end latency  
- Independent retrieval per concept improves recall and interpretability  
- More stable and controllable than prompt-based multi-step rewriting  

---

## Limitations

- Still dependent on index coverage  
  (missing documents such as Llama 3.1 can lead to partial hallucination)  
- Grounding is not strictly enforced unless additional safeguards are added  

---

## Key Insight

SubQuestionQueryEngine provides a **more reliable and production-ready multi-hop RAG architecture** than impl

# HyDE Transform


HyDE (Hypothetical Document Embeddings) Query Transform

HyDE (Hypothetical Document Embeddings) is a query transformation technique that improves retrieval by first asking an LLM to generate a hypothetical answer/document for the query, then embedding and retrieving based on that generated text instead of (or in addition to) the original query.

Example:

Instead of embedding the short user question: “Write about Llama 3.1 Model, BERT and PEFT”

HyDE generates a longer, content-rich “fake answer” describing these topics and uses that text to query the vector index.

In [29]:
query_engine = vector_index.as_query_engine()

In [31]:
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.query_engine.transform_query_engine import TransformQueryEngine

hyde = HyDEQueryTransform(include_original=True)   # True for using both original query and the generated hypothetical document
hyde_query_engine = TransformQueryEngine(query_engine, hyde)

Pipeline:

User query --> LLM generates hypothetical document (HyDE) --> (Optional) combine with original query --> Embed transformed text --> Vector retrieval --> Normal RAG answer

In [32]:
response = hyde_query_engine.query("Write about Llama 3.1 Model, BERT and PEFT")

In [33]:
response.response

'I can only answer using the provided excerpts. The available context discusses Llama (LLaMA) models, PEFT methods (including Llama-Adapter and LoRA), and related tooling; it does not contain information about a "Llama 3.1" model or detailed new specifics about BERT beyond general model-class structure. Below I summarize what is supported by the excerpts.\n\nLLaMA and PEFT\n- LLaMA is presented as a foundational model family with associated classes and tokenizers (LlamaConfig, LlamaTokenizer / LlamaTokenizerFast) and model variants (LlamaModel, LlamaForCausalLM, LlamaForSequenceClassification, LlamaForQuestionAnswering, LlamaForTokenClassification). Flax implementations (FlaxLlamaModel, FlaxLlamaForCausalLM) are also noted.\n- Two parameter-efficient fine-tuning (PEFT) approaches are described:\n  - Llama-Adapter: A PEFT method that keeps the base LLaMA model frozen and learns a small set of adaptation prompts that are prepended to input tokens at higher transformer layers. It uses a z

In [34]:
for src in response.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 e1e3e842-7160-40c4-8e74-772fb8254f5e
Text	 # Llama-Adapter[Llama-Adapter](https://hf.co/papers/2303.16199) is a PEFT method specifically designed for turning Llama into an instruction-following model. The Llama model is frozen and only a set of adaptation prompts prefixed to the input instruction tokens are learned. Since randomly initialized modules inserted into the model can cause the model to lose some of its existing knowledge, Llama-Adapter uses zero-initialized attention with zero gating to progressively add the instructional prompts to the model.The abstract from the paper is:*We present LLaMA-Adapter, a lightweight adaption method to efficiently fine-tune LLaMA into an instruction-following model. Using 52K self-instruct demonstrations, LLaMA-Adapter only introduces 1.2M learnable parameters upon the frozen LLaMA 7B model, and costs less than one hour for fine-tuning on 8 A100 GPUs. Specifically, we adopt a set of learnable adaption prompts, and prepend them to the in

In [35]:
query_bundle = hyde("Write about Llama 3.1 Model, BERT and PEFT")

In [36]:
hyde_doc = query_bundle.embedding_strs[0]

In [37]:
hyde_doc

'Llama 3.1, BERT, and PEFT represent three important and complementary developments in modern NLP: large generative LLMs, foundational encoder models for understanding, and techniques for efficient fine-tuning. Below is a concise but detailed overview of each and how they relate.\n\nLlama 3.1\n- Purpose and architecture: Llama 3.1 is a family member in the Llama series of large language models (LLMs) designed for high-quality text generation, instruction following, and conversational AI. Like other Llama models, it uses a transformer-based decoder (autoregressive) architecture optimized for scaling and efficient inference.\n- Training data and scale: Llama 3.1 is trained on a very large, diverse mixture of web text, code, books, and other curated corpora to capture broad world knowledge and linguistic patterns. Compared to earlier Llama versions, 3.1 typically benefits from updated pretraining data, architecture or training recipe improvements, and refinement to reduce toxicity and fac

# HyDE (Hypothetical Document Embeddings) – Query Transform Summary

## Observed Behavior

From the experiment:

- Retrieved only **PEFT / LLaMA-Adapter / LoRA** documents  
- No **BERT** or **Llama 3.1** documents (they do not exist in the index)  
- The answer correctly stated:

  > “I can only answer using the provided information…”

This shows:

- HyDE improved retrieval focus  
- HyDE cannot retrieve content that is not present in the index  
- Grounding was fully preserved (no hallucination)

**Result:**
- 🟢 Grounding preserved  
- 🟡 Recall limited by corpus coverage  

---

## Strengths

- Improves semantic recall without multi-step reasoning  
- Simple to integrate into any query engine  
- No loop or agent complexity  
- Maintains strict RAG grounding  
- Very effective for dense vector retrieval  

---

## Limitations

- Cannot solve missing corpus coverage  
  (no BERT / Llama 3.1 documents → no retrieval)  
- Depends on the quality of the hypothetical document generated by the LLM  
- Does not decompose multi-entity queries (single retrieval pass only)  

---

## When to Use HyDE

### Best Use Cases

- Short or underspecified queries  
- Domain-specific wording mismatch between query and documents  
- Improving recall in vector-only RAG systems  
- First-stage retriever enhancement before hybrid retrieval or reranking  

### Not Ideal For

- Multi-entity comparison queries  
- Multi-hop reasoning tasks  
- Situations requiring query decomposition  
  (use `SubQuestionQueryEngine` instead)  

---

## Key Insight

HyDE is a **retrieval enhancement technique**, not a reasoning technique.

It improves **what to retrieve**, not **how to reason**.

---

## One-Line Summary

**HyDE expands the query by generating a hypothetical answer to improve semantic retrieval while preserving strict grounding, but it cannot compensate for missing corpus coverage or perform multi-step reasoning.**
